In [1]:
from ib_insync import *
import pandas as pd
import time
import os

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 4002, clientId=2)

master = pd.read_parquet('../data/parquet/instrument_master_sp500.parquet', engine='fastparquet')
print(f"✅ Connecté + {len(master)} actions chargées")

✅ Connecté + 503 actions chargées


In [4]:
ib.reqMarketDataType(3)

contracts = [Stock(row['symbol'], row['exchange'], row['currency']) for _, row in master.iterrows()]

all_spots = []

for i in range(0, len(contracts), 50):
    batch = contracts[i:i+50]
    ib.qualifyContracts(*batch)
    
    tickers = []
    for c in batch:
        t = ib.reqMktData(c, '', False, False)
        tickers.append(t)
    
    ib.sleep(5)
    
    for t in tickers:
        all_spots.append({
            'symbol': t.contract.symbol,
            'conId': t.contract.conId,
            'bid': t.bid if t.bid > 0 else None,
            'ask': t.ask if t.ask > 0 else None,
            'last': t.last if t.last > 0 else None,
            'close': t.close if t.close > 0 else None,
            'volume': t.volume if t.volume >= 0 else None,
            'marketDataType': t.marketDataType,
        })
        ib.cancelMktData(t.contract)
    
    filled = sum(1 for s in all_spots[-len(batch):] if s.get('last') or s.get('close'))
    print(f"Batch {i//50 + 1}/{len(contracts)//50 + 1}: {filled}/{len(batch)} avec prix")
    time.sleep(1)

spots = pd.DataFrame(all_spots)
print(f"\n✅ {len(spots)} actions récupérées")
print(f"Avec last: {spots['last'].notna().sum()}")
print(f"Avec close: {spots['close'].notna().sum()}")
spots.head(10)

Batch 1/11: 50/50 avec prix
Batch 2/11: 50/50 avec prix
Batch 3/11: 50/50 avec prix
Batch 4/11: 50/50 avec prix
Batch 5/11: 50/50 avec prix
Batch 6/11: 50/50 avec prix
Batch 7/11: 50/50 avec prix
Batch 8/11: 50/50 avec prix
Batch 9/11: 50/50 avec prix
Batch 10/11: 50/50 avec prix
Batch 11/11: 3/3 avec prix

✅ 503 actions récupérées
Avec last: 503
Avec close: 503


,symbol,conId,bid,ask,last,close,volume,marketDataType
0,A,1715006,129.37,129.64,129.50,131.62,1.657846e+11,3
1,AAPL,265598,290.16,290.21,290.18,291.58,7.430378e+12,3
2,ABBV,118089500,224.44,224.68,224.62,224.95,7.661808e+11,3
3,ABNB,459530964,128.44,128.67,128.57,129.10,3.843146e+11,3
4,ABT,4065,89.55,89.57,89.56,89.17,1.466850e+12,3
5,ACGL,10763362,91.96,92.08,92.03,91.31,2.060154e+11,3
6,ACN,67889930,166.90,167.15,167.05,170.50,7.173523e+11,3
7,ADBE,265768,223.25,223.33,223.33,233.38,2.456624e+11,3
8,ADI,4157,405.85,406.84,406.35,392.67,5.233613e+11,3
9,ADM,4165,80.74,80.84,80.77,81.28,3.853008e+10,3


In [5]:
# Test sur AAPL

aapl = Stock('AAPL', 'SMART', 'USD')
ib.qualifyContracts(aapl)

# Récupérer le spot AAPL pour cibler les strikes ATM
spot_aapl = spots[spots['symbol'] == 'AAPL']['last'].values[0]
print(f"AAPL spot: {spot_aapl}")

chains = ib.reqSecDefOptParams(aapl.symbol, '', aapl.secType, aapl.conId)
chain = next(c for c in chains if c.exchange == 'SMART')

print(f"Expirations: {len(chain.expirations)}")
print(f"Strikes: {len(chain.strikes)}")

expirations = sorted(chain.expirations)[:3]
strikes = [s for s in sorted(chain.strikes) if spot_aapl * 0.90 <= s <= spot_aapl * 1.10]

print(f"\n3 premières expirations: {expirations}")
print(f"Strikes ATM ±10%: {len(strikes)} strikes")
print(f"Range: {min(strikes)} → {max(strikes)}")

AAPL spot: 290.18
Expirations: 26
Strikes: 123

3 premières expirations: ['20260612', '20260615', '20260617']
Strikes ATM ±10%: 23 strikes
Range: 262.5 → 317.5


In [ ]:
expirations = sorted(chain.expirations)
expirations = [e for e in expirations if e >= '20260620'][:3]
print(f"Expirations choisies: {expirations}")

option_contracts = []
for exp in expirations:
    for strike in strikes:
        for right in ['C', 'P']:
            option_contracts.append(
                Option('AAPL', exp, strike, right, 'SMART', multiplier='100')
            )

print(f"Contrats à qualifier: {len(option_contracts)}")

qualified = []
for i in range(0, len(option_contracts), 50):
    batch = option_contracts[i:i+50]
    ib.qualifyContracts(*batch)
    qualified.extend([c for c in batch if c.conId > 0])
    time.sleep(0.5)

print(f"✅ {len(qualified)} contrats qualifiés sur {len(option_contracts)}")

option_data = []
for i in range(0, len(qualified), 50):
    batch = qualified[i:i+50]
    tickers = [ib.reqMktData(c, '', False, False) for c in batch]
    ib.sleep(5)
    
    for t in tickers:
        option_data.append({
            'symbol': t.contract.symbol,
            'expiry': t.contract.lastTradeDateOrContractMonth,
            'strike': t.contract.strike,
            'right': t.contract.right,
            'bid': t.bid if t.bid > 0 else None,
            'ask': t.ask if t.ask > 0 else None,
            'last': t.last if t.last > 0 else None,
        })
        ib.cancelMktData(t.contract)
    
    print(f"Batch {i//50 + 1}: {len(batch)} options traitées")

opts = pd.DataFrame(option_data)
print(f"\n✅ {len(opts)} options récupérées")
print(f"Avec bid: {opts['bid'].notna().sum()}")
opts.head(10)

Expirations choisies: ['20260622', '20260624', '20260626']
Contrats à qualifier: 138


Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20260626', strike=262.5, right='C', multiplier='100', exchange='SMART')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20260626', strike=262.5, right='P', multiplier='100', exchange='SMART')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20260626', strike=267.5, right='C', multiplier='100', exchange='SMART')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20260626', strike=267.5, right='P', multiplier='100', exchange='SMART')


✅ 134 contrats qualifiés sur 138
Batch 1: 50 options traitées
Batch 2: 50 options traitées
Batch 3: 34 options traitées

✅ 134 options récupérées
Avec bid: 122


,symbol,expiry,strike,right,bid,ask,last
0,AAPL,20260622,262.5,C,27.80,29.95,NaN
1,AAPL,20260622,262.5,P,0.24,0.41,NaN
2,AAPL,20260622,265.0,C,25.70,27.20,NaN
3,AAPL,20260622,265.0,P,0.30,0.46,NaN
4,AAPL,20260622,267.5,C,23.25,24.50,NaN
5,AAPL,20260622,267.5,P,0.38,0.50,0.49
6,AAPL,20260622,270.0,C,21.20,22.10,NaN
7,AAPL,20260622,270.0,P,0.42,0.64,NaN
8,AAPL,20260622,272.5,C,18.85,19.70,NaN
9,AAPL,20260622,272.5,P,0.62,0.79,0.70


In [10]:
today = pd.Timestamp.now().strftime('%Y-%m-%d')

opts.to_parquet(f'../data/raw/options_AAPL_{today}.parquet', index=False, engine='fastparquet')
print(f"✅ Options AAPL sauvegardées: {len(opts)} contrats")

spots.to_parquet(f'../data/raw/spots_{today}.parquet', index=False, engine='fastparquet')
print(f"✅ Sauvegardé: spots_{today}.parquet — {len(spots)} actions")

✅ Options AAPL sauvegardées: 134 contrats
✅ Sauvegardé: spots_2026-06-11.parquet — 503 actions
